**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/liu-2026-001752"], check=True)
    os.chdir("/content/encore/entries/liu-2026-001752")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Liu et al. (2026), Fig. 5a-c, from dandiset 001752

Paper. Liu, Huang, Shen, Shao, Jing et al. (2026), "An atlas of primate insular cortex reveals a
signal-processing strategy in von Economo neurons", Nature Cell Biology 28, 1742-1757,
doi:10.1038/s41556-026-02009-4. The claims file (`claims/001752/10.1038-s41556-026-02009-4.yaml`) was
written from the version of record (Europe PMC, PMC13457102): main text, figure legends, Extended Data
legends and Methods.

Dataset. DANDI:001752, "Electrophysiological Dataset of Neuronal Post-Synaptic Currents in the Primate
Anterior Insular Cortex", version 0.260825.0222 (doi:10.48324/dandi.001752/0.260825.0222). The paper
cites the dandiset without a version, and this is its only published version (the draft has the same 84
assets and size), so it is pinned.

Outside inputs (`with_labels`). The NWB files carry no cell type. The paper's code,
github.com/RFLiu2021/AIC_proj at commit 44cd8a6dfbb1f1ce363d73742e4f3652086b0f9d, holds a table of the
deposited files with their cell types (`ephysio/data_ephy/rawdata_ephysio/metadata_PSC.csv`) and the
paper's per-cell frequencies and amplitudes (`ephysio/data_ephy/sPSCs_Fig7ab.xlsx`,
`ephysio/data_ephy/miniEPSCs_Fig7ab.xlsx`). The table supplies the labels; the sheets are used only to check
the link (claim `psc-file-labels-link`) and as the variant labelling of the mEPSC cells that the claims
file asks for. All measurements are made here from the NWB traces.

| claim | tested here |
|---|---|
| psc-file-labels-link (link, added by this attempt) | yes |
| fig5b-sepsc-frequency, fig5b-sipsc-frequency, fig5b-mepsc-frequency, fig5b-mipsc-frequency | yes |
| fig5c-sepsc-amplitude, fig5c-sipsc-amplitude, fig5c-mepsc-amplitude, fig5c-mipsc-amplitude | yes |
| the paper's other claims (`data: elsewhere`) | not here: their data are in other archives; see `elsewhere_liu_2026.ipynb` and the attempt record |

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import hashlib

import psc

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 40)

meta = psc.version_metadata()
asset_list = psc.assets()
print("dandiset", psc.DANDISET, "version", psc.VERSION, "doi", meta.get("doi"))
print("name:", meta["name"])
print("description:", meta["description"])
print("related:", [(r.get("relation"), r.get("url")) for r in meta.get("relatedResource", [])])
print("assets:", len(asset_list), "total bytes:", sum(a["size"] for a in asset_list))
print("summary:", {k: meta["assetsSummary"].get(k) for k in ("numberOfSubjects", "numberOfFiles", "variableMeasured")})

In [ ]:
recs = psc.read_all(asset_list)
info = pd.DataFrame([{k: v for k, v in r.items() if k != "trace_pA"} for r in recs])
info["obj"] = info.file.str.extract(r"obj-([a-z0-9]+)_")[0]
info["experiment"] = info.session_description.map(psc.experiment_of)
traces = {r["file"]: r["trace_pA"] for r in recs}
print(info[["obj", "subject", "session", "experiment", "cell_id", "rate", "unit", "n_samples", "sex", "age"]].to_string())

## Data check

What each file holds: one `VoltageClampSeries` (current in amperes, one sweep, starting at 0 s) and a
`VoltageClampStimulusSeries` of three zeros (no stimulus). There are no trial tables, intervals or
`intracellular_recordings` tables, and no cell type anywhere in the file. The holding potential is given
only in the session description. The checks below are of the experiment, durations, sampling rate,
holding current, padding, duplicates and noise.

In [ ]:
print("Files by experiment, subject and session (from session_description):")
print(info.groupby(["experiment", "subject", "session"]).size().to_string())
print()
print("Session descriptions:")
print(info.groupby(["experiment", "session_description", "experiment_description", "stimulus_description"]).size().to_string())
print()
print("rates (Hz):", sorted(info.rate.unique()), " units:", sorted(info.unit.unique()),
      " stimulus values:", sorted({v for s in info.stimulus_values for v in s}))
print("sex:", info.sex.value_counts().to_dict(), " age:", info.age.value_counts().to_dict())
print("cell_id is unique within a session?",
      not info.duplicated(["session", "experiment", "cell_id"]).any())
dup_ids = info[info.duplicated(["session", "experiment", "cell_id"], keep=False)].sort_values("cell_id")
print(dup_ids[["obj", "session", "experiment", "cell_id"]].to_string())

In [ ]:
# Padding: trailing runs of exact zeros; holding current; duration; duplicates.
rows = []
for f, x in traces.items():
    xt, cut = psc.trim_padding(x)
    rate = info.set_index("file").rate[f]
    z = xt - psc.baseline(xt, rate)
    rows.append(dict(file=f, cut_zero_samples=cut, duration_s=len(xt) / rate,
                     holding_median_pA=np.median(xt), holding_p1=np.percentile(xt, 1), holding_p99=np.percentile(xt, 99),
                     noise_pA=psc.robust_sd(z),
                     md5=hashlib.md5(x.tobytes()).hexdigest(), seg_md5=hashlib.md5(xt[1000:6000].tobytes()).hexdigest()))
chk = pd.DataFrame(rows)
info = info.merge(chk, on="file")
print("files with trailing zero padding:", (info.cut_zero_samples > 0).sum(),
      "; samples cut per file:", sorted(info.cut_zero_samples[info.cut_zero_samples > 0].unique()))
print(info.groupby("experiment").agg(n=("file", "size"), dur_min=("duration_s", "min"), dur_median=("duration_s", "median"),
                                     dur_max=("duration_s", "max"), under_5min=("duration_s", lambda d: int((d < 300).sum())),
                                     hold_min=("holding_median_pA", "min"), hold_max=("holding_median_pA", "max"),
                                     noise_min=("noise_pA", "min"), noise_median=("noise_pA", "median"),
                                     noise_max=("noise_pA", "max"), noise_over_5pA=("noise_pA", lambda d: int((d > 5).sum())))
      .round(2).to_string())
print("identical traces (whole file):", int(info.md5.duplicated().sum()),
      "; identical 5 s segments at 1 s:", int(info.seg_md5.duplicated().sum()))

In [ ]:
# Near-duplicates: correlation of the first 60 s, referred to the running median (so that the slow drift of
# the holding current, common to many files, does not dominate), between every pair of files of an experiment.
near = []
for exp, g in info.groupby("experiment"):
    fs = list(g.file)
    for i in range(len(fs)):
        for j in range(i + 1, len(fs)):
            a, b = traces[fs[i]][:60000], traces[fs[j]][:60000]
            a, b = a - psc.baseline(a, 1000.0), b - psc.baseline(b, 1000.0)
            n = min(len(a), len(b))
            r = np.corrcoef(a[:n], b[:n])[0, 1]
            near.append((exp, fs[i].split("obj-")[1][:8], fs[j].split("obj-")[1][:8], r))
near = pd.DataFrame(near, columns=["experiment", "a", "b", "r"])
print("pairs compared:", len(near), "; largest correlations:")
print(near.sort_values("r", ascending=False).head(5).round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, (exp, g) in zip(axes.ravel(), info.groupby("experiment")):
    f = g.sort_values("noise_pA").file.iloc[len(g) // 2]
    x, _ = psc.trim_padding(traces[f])
    t = np.arange(len(x)) / 1000.0
    s = slice(60000, 62000)
    ax.plot(t[s], x[s], lw=0.6)
    ax.set_title(f"{exp}  {f.split('obj-')[1][:8]} (median-noise file)")
    ax.set_xlabel("s"); ax.set_ylabel("pA")
plt.tight_layout(); plt.show()
print("plotted: 2 s of one file per experiment; y ranges:",
      {exp: (round(float(psc.trim_padding(traces[g.sort_values('noise_pA').file.iloc[len(g)//2]])[0][60000:62000].min()), 1),
             round(float(psc.trim_padding(traces[g.sort_values('noise_pA').file.iloc[len(g)//2]])[0][60000:62000].max()), 1))
       for exp, g in info.groupby("experiment")})

### Events, under every reading

For each file, events are detected (helper `psc.detect`) in the direction set by the holding potential:
outward for the sIPSC files, held at +20 mV by their description, and inward for the others. Each file is
run with the two threshold readings of the claims file, (a) 5 pA and (b) the recording's RMS noise, over
the whole recording and over its first 30 s (the code's window). A third threshold, three times the
recording's noise, is a variant added here (a common practice in event detection, not the paper's) to
show how far the outcomes depend on detection; it decides nothing.

In [ ]:
SIGN = {"sEPSC": -1, "mEPSC": -1, "mIPSC": -1, "sIPSC": +1}
READINGS = {"5pA_whole": (5.0, None), "rms_whole": (None, None), "5pA_30s": (5.0, 30.0), "rms_30s": (None, 30.0),
            "3rms_whole": ("3rms", None)}
det_rows, detections = [], {}
for _, r in info.iterrows():
    x, _ = psc.trim_padding(traces[r.file])
    for name, (thr, win) in READINGS.items():
        if thr == "3rms":  # variant: three times the recording's noise, a common practice, not the paper's
            thr = 3 * psc.robust_sd(x - psc.baseline(x, r.rate))
        d = psc.detect(x, r.rate, SIGN[r.experiment], thr, win)
        detections[(r.file, name)] = d
        det_rows.append(dict(file=r.file, reading=name, n_events=d["n_events"], n_artifacts=d["n_artifacts"],
                             duration_s=d["duration_s"], freq=d["frequency_hz"], amp=d["amplitude_pa"],
                             threshold=d["threshold_pa"]))
det = pd.DataFrame(det_rows)
wide = det.pivot(index="file", columns="reading", values=["freq", "amp", "n_events", "n_artifacts", "threshold"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()
info = info.merge(wide, on="file")
print("events left out as artefacts (> 500 pA), whole recording, 5 pA:",
      info[info.n_artifacts_5pA_whole > 0][["obj", "experiment", "n_artifacts_5pA_whole"]].to_dict("records"))
print("files with fewer than 200 events (5 pA, whole):", int((info.n_events_5pA_whole < 200).sum()), "of", len(info))
print(info.groupby("experiment")[["freq_5pA_whole", "amp_5pA_whole", "freq_rms_whole", "amp_rms_whole", "threshold_rms_whole"]]
      .describe().T.round(2).to_string())

In [ ]:
# Time base: the decay of the average isolated event (peak to 37%), at the stated 1000 Hz.
dec = []
for _, r in info.iterrows():
    x, _ = psc.trim_padding(traces[r.file])
    d = detections[(r.file, "5pA_whole")]
    dec.append(psc.decay_time_ms(x, r.rate, SIGN[r.experiment], d["peaks"], d["amplitudes"]))
info["decay_ms"] = dec
print("decay time of the average event (ms), by experiment:")
print(info.groupby("experiment").decay_ms.describe().round(1).to_string())

### Detector checks

Two checks of the detector, asked for by the verifier. (1) A null from the data: peaks of the opposite
polarity. Noise is symmetric about the baseline and synaptic events are not, so the rate of peaks detected in
the wrong direction estimates how many false events each threshold admits (it overestimates a little, as the
decay of a real event can make an opposite peak against the baseline before it). (2) A positive control:
synthetic events of known times and amplitudes (`psc.inject_events`, 2 Hz, log-normal amplitudes of median
10 pA, the experiment's decay) are added to every real trace, and the share each reading recovers within
3 ms is counted. Neither decides an outcome.

In [ ]:
# Detector checks. (1) A null from the data: events of the opposite polarity. Noise is symmetric about the
# baseline and synaptic events are not, so peaks detected in the wrong direction estimate the false events
# each reading admits. (2) A positive control: synthetic events of known times and amplitudes (helper
# psc.inject_events) added to every real trace, and the share of them each reading recovers.
rng_check = np.random.default_rng(20261103)
decay_by_exp = info.groupby("experiment").decay_ms.median()
chk_rows = []
for _, r in info.iterrows():
    x, _ = psc.trim_padding(traces[r.file])
    s = SIGN[r.experiment]
    y, t_inj, a_inj = psc.inject_events(x, r.rate, s, decay_by_exp[r.experiment], rng_check)
    row = dict(file=r.file, experiment=r.experiment, noise_pA=r.noise_pA, n_injected=len(t_inj))
    for name, thr in (("5pA", 5.0), ("rms", None)):
        row[f"opp_freq_{name}"] = psc.detect(x, r.rate, -s, thr)["frequency_hz"]
        d = psc.detect(y, r.rate, s, thr)
        hit = psc.recovered(d["peaks"], t_inj, int(0.003 * r.rate))
        row[f"hit_{name}"] = hit.mean()
        row[f"added_freq_{name}"] = d["frequency_hz"] - r[f"freq_{name}_whole"]
        row[f"hit_over10pA_{name}"] = hit[a_inj >= 10].mean()
    chk_rows.append(row)
chk2 = pd.DataFrame(chk_rows)
info = info.merge(chk2.drop(columns=["experiment", "noise_pA"]), on="file")
print(f"injected: {psc.INJECT_RATE_HZ} Hz, amplitudes log-normal, median {psc.INJECT_MEDIAN_PA} pA "
      f"(5 to 95%: {psc.INJECT_MEDIAN_PA*np.exp(-1.645*psc.INJECT_SIGMA):.1f} to {psc.INJECT_MEDIAN_PA*np.exp(1.645*psc.INJECT_SIGMA):.1f} pA)")
info["noise_bin"] = pd.cut(info.noise_pA, [0, 2, 4, 6, 100], labels=["<2", "2-4", "4-6", ">6"])
summ = info.groupby(["experiment", "noise_bin"], observed=True).agg(
    files=("file", "size"), freq_5pA=("freq_5pA_whole", "median"), opposite_5pA=("opp_freq_5pA", "median"),
    freq_rms=("freq_rms_whole", "median"), opposite_rms=("opp_freq_rms", "median"),
    recovered_5pA=("hit_5pA", "median"), recovered_10pA_up_5pA=("hit_over10pA_5pA", "median"),
    added_5pA=("added_freq_5pA", "median"), recovered_rms=("hit_rms", "median"))
print("medians per experiment and noise (pA): frequency (Hz) detected in the events' direction and in the opposite one,")
print("share of injected events recovered (all, and those of 10 pA or more), and the frequency added by the injection:")
print(summ.round(2).to_string())
for name in ("5pA", "rms"):
    r_ = stats.spearmanr(info.noise_pA, info[f"opp_freq_{name}"])
    print(f"{name}: opposite-polarity frequency against noise, Spearman rho {r_[0]:.2f}; its share of the frequency "
          f"in the events' direction, median {np.median(info[f'opp_freq_{name}'] / info[f'freq_{name}_whole']):.2f}")

What the checks show. At 5 pA, in the files with noise below 2 pA (all 15 mEPSC files, 34 of 36 mIPSC,
10 of 18 sEPSC), opposite-polarity peaks are rare (medians 0.06 to 0.44 Hz, against 0.73 to 2.43 Hz in the
events' direction) and 91 to 94% of the injected events are recovered. In noisier files the opposite rate
grows with the noise. In the sEPSC files with noise of 2 to 4 pA it is about half the forward one (7.3
against 13.2 Hz; per-file ratio median 0.56), so a large part of what is detected there is noise. In the
sIPSC files it is close to or above the forward one (per-file ratio median 0.94 at 2 to 4 pA, 1.09 at 4 to
6 pA and 1.24 above 6 pA, where the rates are 34.0 against 27.4 Hz), except in two files, w9b4ds (0.43) and
1tsbdsc (0.59), where it is about half (cell after the fig5b-sipsc-frequency table); no sIPSC file has noise
below 2 pA. At the
RMS threshold, reading (b), the opposite rate equals the forward one in every experiment (median ratio
1.05; mEPSC 28.2 against 27.3 Hz), so that reading counts noise, not events.

## psc-file-labels-link

> The dandiset's NWB files are the recordings of Fig. 5a-c, and the code's table of deposited files gives
> each its cell type, consistent with the per-cell values of the code's analysis sheets.

Records are matched by NWB file name (`rawFileName` of metadata_PSC.csv); the table's original file names
are matched to the cell names of the sheets by date, slice and cell (`psc.name_key`).

In [ ]:
lab = psc.labels()
lab["key"] = [psc.name_key(n, e) for n, e in zip(lab.filename_original, lab.Experiment)]
files = set(info.file)
print("table rows:", len(lab), "; distinct files named:", lab.rawFileName.nunique(),
      "; rows naming a dandiset file:", int(lab.rawFileName.isin(files).sum()))
print("dandiset files the table does not name:", sorted(files - set(lab.rawFileName)))
print("files named by more than one row:")
print(lab[lab.rawFileName.duplicated(keep=False)][["filename_original", "rawFileName", "cellType", "Experiment"]].to_string())
x = lab.merge(info[["file", "experiment", "subject"]], left_on="rawFileName", right_on="file", how="left")
print("table experiment equals the file's own description:", bool((x.Experiment == x.experiment).all()),
      "; table subject equals the file's:", bool((x.Subject == x.subject).all()))
counts = lab.groupby(["Experiment", "cellType"]).size().unstack()
counts["paper_VEN"] = pd.Series({"sEPSC": 8, "sIPSC": 7, "mEPSC": 8, "mIPSC": 15})
counts["paper_PC"] = pd.Series({"sEPSC": 11, "sIPSC": 8, "mEPSC": 8, "mIPSC": 21})
print("rows per experiment and cell type, against the Fig. 5b legend:")
print(counts.to_string())

In [ ]:
# The rows of the code's sheets that psc.code_tables reads, as they are in the sheets (header rows and the first entries).
for name, sh, rr, cc in ((psc.SPSC_SHEET, "summary", slice(0, 5), slice(0, 14)), (psc.SPSC_SHEET, "summary", slice(18, 23), slice(0, 14)),
                         (psc.MINI_SHEET, "mEPSC_sum", slice(0, 4), slice(0, 8)), (psc.MINI_SHEET, "mIPSC_sum", slice(0, 4), slice(0, 8))):
    s = psc.sheet(name, sh)
    print(f"{name} / {sh}, rows {rr.start} to {rr.stop - 1} (of {len(s)}), columns {cc.start} to {cc.stop - 1} (of {s.shape[1]}):")
    print(s.iloc[rr, cc].to_string(), "\n")

In [ ]:
tab = psc.code_tables().rename(columns={"experiment": "Experiment"})
print("sheet entries per experiment and sheet cell type:")
print(tab.groupby(["Experiment", "sheet_type"]).size().unstack().to_string())
lab1 = lab.drop_duplicates(["key", "Experiment"])
link = tab.merge(lab1[["key", "Experiment", "rawFileName", "cellType", "filename_original"]], on=["key", "Experiment"], how="left")
print("sheet entries with no table row:", link[link.rawFileName.isna()][["Experiment", "sheet_name", "sheet_type"]].to_dict("records"))
link = link.dropna(subset=["rawFileName"])
link["same_type"] = link.sheet_type == link.cellType
print("sheet type equals table type, by experiment:")
print(link.groupby("Experiment").same_type.agg(["sum", "size"]).to_string())
print(link[~link.same_type][["Experiment", "sheet_name", "sheet_type", "filename_original", "cellType"]].to_string())

In [ ]:
link = link.merge(info[["file", "obj", "subject", "freq_5pA_whole", "amp_5pA_whole", "n_events_5pA_30s", "freq_5pA_30s",
                        "noise_pA"]], left_on="rawFileName", right_on="file")
print("Per experiment: Spearman correlation of the frequency and amplitude measured here (5 pA, whole recording)")
print("with the sheet's, and the median ratio here/sheet:")
agree = []
for exp, g in link.groupby("Experiment"):
    rf = stats.spearmanr(g.freq_5pA_whole, g.sheet_freq); ra = stats.spearmanr(g.amp_5pA_whole, g.sheet_amp)
    agree.append(dict(experiment=exp, n=len(g), rho_freq=rf[0], p_freq=rf[1], rho_amp=ra[0], p_amp=ra[1],
                      ratio_freq=np.median(g.freq_5pA_whole / g.sheet_freq), ratio_amp=np.median(g.amp_5pA_whole / g.sheet_amp)))
agree = pd.DataFrame(agree)
print(agree.round(3).to_string())
print()
print("by subject (median ratio of frequency here/sheet, and median noise):")
print(link.groupby(["Experiment", "subject"]).apply(lambda g: pd.Series(dict(
    n=len(g), ratio_freq=np.median(g.freq_5pA_whole / g.sheet_freq), ratio_amp=np.median(g.amp_5pA_whole / g.sheet_amp),
    noise=np.median(g.noise_pA)))).round(2).to_string())
mi = link[link.Experiment == "mIPSC"]
r30 = stats.spearmanr(mi.n_events_5pA_30s, mi.sheet_n_events)
print(f"\nmIPSC, events in the first 30 s here against the sheet's nEvents over 30 s: rho = {r30[0]:.2f} (p = {r30[1]:.2g}), "
      f"median ratio {np.median(mi.n_events_5pA_30s / mi.sheet_n_events):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (exp, g) in zip(axes, link.groupby("Experiment")):
    ax.scatter(g.sheet_freq, g.freq_5pA_whole, c=np.where(g.cellType == "VEN", "tab:red", "tab:blue"), s=18)
    lim = max(g.sheet_freq.max(), g.freq_5pA_whole.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k:", lw=0.8)
    ax.set_title(exp); ax.set_xlabel("sheet frequency (Hz)"); ax.set_ylabel("here, 5 pA, whole (Hz)")
plt.tight_layout(); plt.show()
print("axes span every point (0 to the larger maximum); red VEN, blue PC by the table")

In [ ]:
# Per file: noise against the frequency detected in the events' direction and in the opposite one, the
# positive control, and the sheet's frequency (where the file is linked to a sheet entry).
pf = info.merge(link[["file", "sheet_freq", "cellType"]].drop_duplicates("file"), on="file", how="left")
cols = ["experiment", "obj", "subject", "cellType", "noise_pA", "duration_s", "freq_5pA_whole", "opp_freq_5pA", "freq_rms_whole",
        "opp_freq_rms", "hit_5pA", "sheet_freq"]
print(pf.sort_values(["experiment", "noise_pA"])[cols].round(2).to_string(index=False))
for exp, g in pf.dropna(subset=["sheet_freq"]).groupby("experiment"):
    lo = g[g.noise_pA < 2]
    r_all = stats.spearmanr(g.freq_5pA_whole, g.sheet_freq)[0]
    r_lo = stats.spearmanr(lo.freq_5pA_whole, lo.sheet_freq)[0] if len(lo) > 3 else np.nan
    print(f"{exp}: rho with the sheet's frequency, all {len(g)} files {r_all:.2f}; files with noise < 2 pA ({len(lo)}) {r_lo:.2f}")

**Result: partially reproduced.** All 85 rows of metadata_PSC.csv name a file of the dandiset, the
table's experiment and subject agree with every file's own metadata, and its counts per experiment and cell
type equal the Fig. 5b legend except mEPSC VENs (7 rows against 8). Three parts fail. (1) One file,
`fwphyw` (sIPSC, Cell_AS011 of session 201126), is named by no row, while `1s3ysck` is named twice, once as
PC 20201126s5c11a and once as VEN 20201126s5c11b; and the sEPSC PC 20201126s1c2 is one file counted twice,
as the code's sheet does. (2) For mEPSC the sheet's cell type differs from the table's for all 15 matched
cells, and the sheet's PC S02C0321 has no row; for sEPSC, sIPSC and mIPSC the types agree (19, 15 and 36
of 36). (3) The frequencies measured here correlate with the sheet's for mIPSC (rho 0.73; 0.87 in its 34
files with noise below 2 pA) but not significantly for mEPSC (rho 0.51, p 0.054), sEPSC (0.34) or sIPSC
(0.19). The frequencies here are near the sheet's in the low-noise sEPSC files of Mfa201023 (median ratio
0.77, amplitude ratio 1.08) and 4.6 to 15 times the sheet's in the noisier sEPSC and sIPSC files of
Mfa201126 and Mfa201030, where the detector check finds opposite-polarity peaks at half the rate of the
events or more (about half in the sEPSC files of 2 to 4 pA noise, about as many in most sIPSC files);
so much of the disagreement is in detection, not in which recording is which. In the 10 sEPSC files with
noise below 2 pA the correlation is still weak (rho 0.16), so detection does not explain all of it. The
mIPSC counts over the first 30 s correlate with the sheet's 30 s counts (rho 0.57, median ratio 0.65).

## Group comparisons: shared code

Per cell, frequency (Hz) and mean amplitude (pA) under each reading. The cell groups are the table's rows
(the dataset's labels, as the claims file asks). The readings the claims file and the data require:

- threshold (a) 5 pA, (b) the recording's RMS noise, over the whole recording (both decide);
  the first 30 s (the code's window) is a variant;
- sEPSC: the PC group as (a) the paper's 11 entries, which count file `15e3e16` (cell 20201126s1c2) twice,
  and (b) the 10 distinct cells;
- sIPSC: the table gives PC 20201126s5c11a and VEN 20201126s5c11b the same file (`1s3ysck`) and names no
  row for `fwphyw`, the other file of cell `Cell_AS011` in that session. Readings: (A) the table as it is,
  one file in both groups; (B) `1s3ysck` is 11a (PC) and `fwphyw` 11b (VEN); (C) the reverse;
- mEPSC: the table's labels decide; the sheet's labelling (the other way round) is a variant.

The test is the two-tailed Wilcoxon rank-sum test as MATLAB's `ranksum` computes it by default, the function
Fig5ab_ana_sPSC.m and Fig5ab_ana_mPSC.m call (helper `psc.ranksum`): exact when the smaller group has fewer than
10 cells and both together fewer than 20 (sEPSC, sIPSC, mEPSC), otherwise the normal approximation (mIPSC).
The column `test` says which.

Each claim also shows `robust(...)`: variants at 5 pA over the whole recording that bear on the recordings'
quality and decide nothing (files with noise below 2 pA; files of at least 5 min and with at least 200 events,
the Methods' minimums; and the frequency minus that of opposite-polarity peaks, the noise null of the detector check).

In [ ]:
def groups(exp, reading_rows="table", s11=None, sheet_labels=False):
    """The rows (file, cell type) of one experiment under a reading."""
    g = lab[lab.Experiment == exp][["filename_original", "rawFileName", "cellType", "key"]].copy()
    if reading_rows == "distinct":
        g = g.drop_duplicates("rawFileName")
    if exp == "sIPSC" and s11 in ("B", "C"):
        other = [f for f in files - set(lab.rawFileName) if "fwphyw" in f][0]
        tgt = "20201126s5c11b+20mV" if s11 == "B" else "20201126s5c11a+20mV"
        g.loc[g.filename_original == tgt, "rawFileName"] = other
    if sheet_labels:
        st = tab[tab.Experiment == exp].set_index("key").sheet_type
        g["cellType"] = g.key.map(st)
        g = g.dropna(subset=["cellType"])
    return g.merge(info, left_on="rawFileName", right_on="file")

def compare(g, col):
    v, p = g[g.cellType == "VEN"][col].dropna(), g[g.cellType == "PC"][col].dropna()
    if len(v) < 2 or len(p) < 2:
        return dict(n_VEN=len(v), n_PC=len(p))
    u, pval, method = psc.ranksum(v, p)
    return dict(n_VEN=len(v), n_PC=len(p), VEN_median=v.median(), PC_median=p.median(),
                VEN_range=f"{v.min():.2f}-{v.max():.2f}", PC_range=f"{p.min():.2f}-{p.max():.2f}",
                diff=v.median() - p.median(), U=u, p=pval, test=method)

def table(exp, measure, extra=()):
    rows = []
    for rd in ("5pA_whole", "rms_whole", "5pA_30s", "rms_30s", "3rms_whole"):
        for opt in extra or [dict()]:
            g = groups(exp, **opt)
            rows.append(dict(reading=rd, **{k: str(v) for k, v in opt.items()}, **compare(g, f"{measure}_{rd}")))
    return pd.DataFrame(rows)

def robust(exp, measure, **opt):
    """Variants at 5 pA over the whole recording that bear on the recordings' quality (they decide nothing):
    all files; files with noise below 2 pA; files of at least 5 min; files with at least 200 events; and,
    for frequency, the frequency minus that of opposite-polarity peaks (the noise null of the detector check)."""
    g = groups(exp, **opt)
    col = f"{measure}_5pA_whole"
    sets = [("all files", g, col), ("noise < 2 pA", g[g.noise_pA < 2], col),
            ("at least 5 min", g[g.duration_s >= 300], col), ("at least 200 events", g[g.n_events_5pA_whole >= 200], col)]
    if measure == "freq":
        g = g.assign(freq_net=g.freq_5pA_whole - g.opp_freq_5pA)
        sets.append(("minus opposite-polarity rate", g, "freq_net"))
    return pd.DataFrame([dict(variant=name, **compare(s, c)) for name, s, c in sets])

def by_subject(exp, col, **opt):
    g = groups(exp, **opt)
    return g.groupby(["subject", "cellType"])[col].agg(["size", "median", "min", "max"]).round(2)


### The paper's own per-cell values

The code's sheets hold the paper's per-cell frequencies and amplitudes. Tested as the figure tests them, with the
sheet's labels and with the table's (the dataset's), they show what the paper's analysis gives and how far the
labels alone change it, apart from any detection. They decide no outcome.

In [ ]:
# The paper's own per-cell values (the code's sheets), tested as the paper's figure tests them, with the
# sheet's labels and with the table's labels (the dataset's). This separates the labels from the detection;
# it decides no outcome.
rows = []
for exp, g in tab.groupby("Experiment"):
    g = g.merge(lab.drop_duplicates(["key", "Experiment"])[["key", "Experiment", "cellType"]], on=["key", "Experiment"], how="left")
    for labelling, col in (("sheet", "sheet_type"), ("table", "cellType")):
        gg = g.dropna(subset=[col]).assign(cellType=lambda d: d[col])
        for m in ("sheet_freq", "sheet_amp"):
            rows.append(dict(experiment=exp, labels=labelling, measure=m, **compare(gg, m)))
sheet_tests = pd.DataFrame(rows)
print(sheet_tests.round(4).to_string())

## fig5b-sepsc-frequency

> The frequency of spontaneous EPSCs is lower in VENs than in neighbouring PCs.

Criterion: reproduced when, under every combination of the readings (threshold 5 pA or RMS noise; PC group
of 11 entries or 10 cells), the VEN median is below the PC median with p < 0.05; blocked as
`methods_underspecified` when the readings disagree.

In [ ]:
t = table("sEPSC", "freq", [dict(reading_rows="table"), dict(reading_rows="distinct")])
print(t.round(4).to_string())
print("\nby subject (5 pA, whole, table rows):")
print(by_subject("sEPSC", "freq_5pA_whole").to_string())
print("\nby subject (RMS, whole, table rows):")
print(by_subject("sEPSC", "freq_rms_whole").to_string())
g = groups("sEPSC")
print("\nnoise (pA) by subject and cell type:")
print(g.groupby(["subject", "cellType"]).noise_pA.agg(["median", "min", "max"]).round(2).to_string())
print("\nquality variants (5 pA, whole), 11 PC entries:")
print(robust("sEPSC", "freq").round(4).to_string())
print("10 distinct PCs:")
print(robust("sEPSC", "freq", reading_rows="distinct").round(4).to_string())

**Result: blocked (methods_underspecified).** The two threshold readings the criterion names disagree.
At 5 pA the VEN median is below the PC median (2.06 against 9.36 Hz, exact rank-sum p = 0.026 with 11 PC
entries; 2.06 against 11.30 Hz, p = 0.027 with 10 cells). At the recording's RMS noise the VEN median is
above the PC median (32.1 against 23.0 Hz, p = 0.091; 10 cells p = 0.146), but the detector check shows that
this reading counts noise: opposite-polarity peaks are as frequent as the events (sEPSC files with noise
below 2 pA, 31.6 against 30.0 Hz). The 30 s window gives the same split (5 pA p = 0.005, RMS VEN higher).
With three times the noise, a variant, VEN 2.82 against PC 3.57 Hz, p = 0.31. The 5 pA difference rests
partly on noise: 8 of the 11 PC rows come from the noisier subject Mfa201126 (median PC noise 3.99 pA). In
the files with noise below 2 pA VENs are still lower (1.53 against 3.38 Hz, 6 and 4 cells, p = 0.038), and
after the opposite-polarity rate is subtracted p = 0.062; only 2 VEN and 2 PC files last 5 min or more
(p = 0.67). The paper's own sheet values give VEN 1.21 against PC 3.25 Hz, p = 0.0068.

## fig5b-sipsc-frequency

> The frequency of spontaneous IPSCs is lower in VENs than in neighbouring PCs.

Events are outward: the files are described as held at +20 mV, and their holding currents are positive
(data check). Criterion as for sEPSC, with no duplicate reading; the 11a/11b readings (A, B, C) come from
the table.

In [ ]:
t = table("sIPSC", "freq", [dict(s11="A"), dict(s11="B"), dict(s11="C")])
print(t.round(4).to_string())
print("\nby subject (5 pA, whole, reading B):")
print(by_subject("sIPSC", "freq_5pA_whole", s11="B").to_string())
print("\nby subject (RMS, whole, reading B):")
print(by_subject("sIPSC", "freq_rms_whole", s11="B").to_string())
for s in "ABC":
    print(f"\nquality variants (5 pA, whole), reading {s}:")
    print(robust("sIPSC", "freq", s11=s).round(4).to_string())

In [ ]:
# Detection in the sIPSC files, the analysis behind the block of the sIPSC claims:
# per file, the noise, the frequency detected at 5 pA in the events' direction and in the opposite one and
# their ratio, the share of injected events recovered, and the frequency and amplitude here over the sheet's.
q = pf.merge(link[["file", "sheet_amp"]].drop_duplicates("file"), on="file", how="left")
q = q.assign(opp_ratio=q.opp_freq_5pA / q.freq_5pA_whole, freq_over_sheet=q.freq_5pA_whole / q.sheet_freq,
             amp_over_sheet=q.amp_5pA_whole / q.sheet_amp)
si = q[q.experiment == "sIPSC"].sort_values("noise_pA")
print(si[["obj", "cellType", "noise_pA", "freq_5pA_whole", "opp_freq_5pA", "opp_ratio", "hit_5pA", "hit_over10pA_5pA",
          "freq_over_sheet", "amp_over_sheet"]].round(2).to_string(index=False))
print(f"\nsIPSC files: {len(si)}; noise below 2 pA: {int((si.noise_pA < 2).sum())}; opposite/forward ratio median "
      f"{si.opp_ratio.median():.2f} ({si.opp_ratio.min():.2f} to {si.opp_ratio.max():.2f}), 0.5 or more in "
      f"{int((si.opp_ratio >= 0.5).sum())}; injected events recovered {si.hit_5pA.min():.2f} to {si.hit_5pA.max():.2f}")
print("\nfor comparison, opposite/forward ratio at 5 pA, median per experiment and noise (pA):")
print(q.groupby(["experiment", "noise_bin"], observed=True).agg(files=("file", "size"), opp_ratio=("opp_ratio", "median"),
      recovered=("hit_5pA", "median")).round(2).to_string())

**Result: blocked, `methods_underspecified`.** In these recordings a detector at the Methods' 5 pA threshold
does not separate events from noise, and the paper gives no other detection setting (miniAnalysis's
amplitude and area thresholds, filtering) that would; the RMS reading counts noise in every experiment. So
no measurement of sIPSC frequency here has a defensible basis. The third verification round agreed. As
written, the criterion would give not reproduced: under neither threshold reading is the VEN median below
the PC median. At
5 pA VEN 19.9 against PC 15.4 Hz (p = 0.34 to 0.40 under the three assignments of file `fwphyw`); at the RMS
noise VEN 13.0 to 13.6 against PC 9.5 to 10.1 Hz (p = 0.0006 to 0.072, VEN higher). With three times the
noise, a variant, VEN 1.88 against PC 2.27 to 2.55 Hz, lower but not significantly (p = 0.23 to 0.54). These
frequencies largely measure noise (cell above): no sIPSC file has noise below 2 pA; in 13 of the 15 files
opposite-polarity peaks are 0.87 to 1.54 times as frequent as the events, and about half as frequent in
the other two (w9b4ds 0.43, 1tsbdsc 0.59); subtracting them leaves medians below zero (p = 0.69 to 0.96);
44 to 83% of injected events are recovered per file; and the 5 pA frequencies are 1.9 to 24 times the
code's sheet values (median 5.5). In the files of at least 5 min VENs are lower (9.9
to 10.6 against 18.5 Hz, 2 or 3 VENs and 4 or 5 PCs, p = 0.11 to 0.38). The paper's own sheet values give
VEN 1.77 against PC 3.25 Hz, p = 0.0093.

## fig5b-mepsc-frequency

> The frequency of miniature EPSCs (in tetrodotoxin) is lower in VENs than in neighbouring PCs.

Criterion: both threshold readings with the table's labels decide; the sheet's labels are a variant.

In [ ]:
t = table("mEPSC", "freq", [dict(sheet_labels=False), dict(sheet_labels=True)])
print(t.round(4).to_string())
print("\nby cell, 5 pA, whole recording (table label, sheet label):")
g = groups("mEPSC").merge(tab[tab.Experiment == "mEPSC"][["key", "sheet_type", "sheet_freq"]], on="key", how="left")
print(g[["filename_original", "obj", "cellType", "sheet_type", "freq_5pA_whole", "sheet_freq", "noise_pA"]].round(2).to_string())
print("\nquality variants (5 pA, whole), table labels:")
print(robust("mEPSC", "freq").round(4).to_string())
print("sheet labels:")
print(robust("mEPSC", "freq", sheet_labels=True).round(4).to_string())

**Result: not reproduced.** With the table's labels (the dataset's) the VEN median is above the PC
median under both threshold readings (5 pA, 0.92 against 0.69 Hz, p = 0.61; RMS, 28.4 against 24.9 Hz,
p = 0.15) and over 30 s (p = 0.34). All mEPSC files have noise below 2 pA, where the detector check recovers
94% of injected events and finds 0.06 Hz of opposite-polarity peaks, so detection is not in doubt at 5 pA.
With the sheet's labels, the variant, the order reverses (0.69 against 0.92 Hz) but is not significant
(p = 0.61). The paper's own sheet frequencies give VEN 1.39 against PC 1.80 Hz (p = 0.028) with the sheet's
labels and the reverse (VEN 1.97 against PC 1.39 Hz, p = 0.029) with the table's: the paper's result follows
from its labelling, and the dataset's labels reverse it. The two labellings are opposite for every mEPSC
cell, and the dandiset cannot tell which is right.

## fig5b-mipsc-frequency

> The frequency of miniature IPSCs (in tetrodotoxin) is lower in VENs than in neighbouring PCs.

In [ ]:
t = table("mIPSC", "freq")
print(t.round(4).to_string())
print("\nby session (AS and HS in the file names; one subject):")
g = groups("mIPSC")
g["sess"] = g.session
print(g.groupby(["sess", "cellType"]).freq_5pA_whole.agg(["size", "median", "min", "max"]).round(2).to_string())
print("\nfiles whose whole-recording frequency exceeds 10 Hz (5 pA):",
      g[g.freq_5pA_whole > 10][["filename_original", "obj", "cellType", "freq_5pA_whole", "noise_pA"]].round(2).to_dict("records"))
print("\nquality variants (5 pA, whole):")
print(robust("mIPSC", "freq").round(4).to_string())

**Result: blocked (methods_underspecified).** The two threshold readings disagree. At 5 pA the VEN median
is below the PC median (1.23 against 2.35 Hz, p = 0.0074); at the RMS noise it is lower but not
significantly (19.0 against 28.9 Hz, p = 0.24), and the detector check shows that this reading counts noise
(mIPSC files with noise below 2 pA, 27.2 Hz of opposite-polarity peaks against 25.9 Hz). Over the first 30 s,
5 pA gives p = 0.075. With three times the noise, a variant, VEN 2.18 against PC 3.68 Hz, p = 0.0005. At 5 pA
the difference holds in the 34 files with noise below 2 pA (p = 0.0026), in the 14 files of at least 5 min
(p = 0.029) and after the opposite-polarity rate is subtracted (p = 0.0064), but not in the 27 files with at
least 200 events (p = 0.13). VENs are lower in both sessions (AS 1.51 against 2.49 Hz, HS 0.90 against 2.04
Hz). The paper's own sheet values give VEN 1.83 against PC 3.00 Hz, p = 0.010.

## fig5c-sepsc-amplitude

> The amplitude of spontaneous EPSCs does not differ significantly between VENs and PCs.

Claim of absence: judged on the two-tailed rank-sum test at 5 pA, whole recording, 8 VEN and 11 PC
entries; reproduced when p >= 0.05. RMS threshold, 10 cells and 30 s are variants.

In [ ]:
t = table("sEPSC", "amp", [dict(reading_rows="table"), dict(reading_rows="distinct")])
print(t.round(4).to_string())
print("\nby subject (5 pA, whole):")
print(by_subject("sEPSC", "amp_5pA_whole").to_string())
print("\nquality variants (5 pA, whole), 11 PC entries:")
print(robust("sEPSC", "amp").round(4).to_string())

**Result: reproduced.** At 5 pA over the whole recording, VEN 8.20 against PC 8.51 pA (difference
-0.31 pA), exact p = 0.24 with the 11 PC entries (10 cells, p = 0.24). The variants differ: the RMS threshold
gives p = 0.012 (VEN 2.77 against 5.86 pA), the 30 s window p = 0.033, and three times the noise p = 0.016,
all with VEN smaller, while the files with noise below 2 pA give p = 0.91. The absence holds under the
paper's 5 pA test as the criterion fixes it, and depends on the window and the threshold. The paper's sheet
amplitudes do not differ (p = 0.11).

## fig5c-sipsc-amplitude

> The amplitude of spontaneous IPSCs does not differ significantly between VENs and PCs.

The decisive test uses the table as it is (reading A), the dataset's labels; B and C are shown.

In [ ]:
t = table("sIPSC", "amp", [dict(s11="A"), dict(s11="B"), dict(s11="C")])
print(t.round(4).to_string())
print("\nby subject (5 pA, whole, reading A):")
print(by_subject("sIPSC", "amp_5pA_whole", s11="A").to_string())
for s in "ABC":
    print(f"\nquality variants (5 pA, whole), reading {s}:")
    print(robust("sIPSC", "amp", s11=s).round(4).to_string())

**Result: blocked, `methods_underspecified`,** for the reason given under fig5b-sipsc-frequency, as the
third verification round agreed. The amplitudes come from the same detection, which does not separate
events from noise in these files; noise peaks just above the 5 pA threshold pull every cell's mean amplitude
towards the same value, which biases the test towards no difference. At 5 pA, whole recording, the table's labels (reading A): VEN 11.8 against PC
14.4 pA, p = 0.094 (B 0.072, C 0.094). The RMS threshold (p = 0.40 to 0.54), the 30 s window (p = 0.054 to
0.072) and three times the noise (p = 0.34 to 0.40) also find no difference. No sIPSC file has noise below
2 pA, and the amplitudes are 0.21 to 0.40 times the sheet's (median per subject, link claim; 0.19 to 0.53
per file, cell after the fig5b-sipsc-frequency table), so the detection here does not measure the paper's
amplitudes; the paper's sheet amplitudes do not differ either (37.6 against 35.9 pA, p = 0.69).

## fig5c-mepsc-amplitude

> The amplitude of miniature EPSCs does not differ significantly between VENs and PCs.

In [ ]:
t = table("mEPSC", "amp", [dict(sheet_labels=False), dict(sheet_labels=True)])
print(t.round(4).to_string())
print("\nquality variants (5 pA, whole), table labels:")
print(robust("mEPSC", "amp").round(4).to_string())

**Result: reproduced.** VEN 8.55 against PC 7.81 pA at 5 pA, p = 0.61, with the table's labels; with the
sheet's labels the same p; every variant has p of 0.34 or more. The paper's sheet amplitudes do not differ
either (p = 0.57 with its labels, 0.87 with the table's).

## fig5c-mipsc-amplitude

> The amplitude of miniature IPSCs does not differ significantly between VENs and PCs.

In [ ]:
t = table("mIPSC", "amp")
print(t.round(4).to_string())
g = groups("mIPSC")
print("\nby session (5 pA, whole):")
print(g.groupby(["session", "cellType"]).amp_5pA_whole.agg(["size", "median", "min", "max"]).round(2).to_string())
print("\nquality variants (5 pA, whole):")
print(robust("mIPSC", "amp").round(4).to_string())

**Result: reproduced.** VEN 9.57 against PC 10.20 pA at 5 pA, p = 0.46; RMS p = 0.70, 30 s p = 0.46,
three times the noise p = 0.35; files with noise below 2 pA p = 0.34, of at least 5 min p = 0.11, with at
least 200 events p = 0.85. The paper's sheet amplitudes do not differ (p = 0.14).

## Summary

| claim | outcome | numbers |
|---|---|---|
| psc-file-labels-link | partially reproduced | 85 of 85 rows name a file; 1 file unnamed, 1 named twice; mEPSC labels opposite to the sheet's (0 of 15 agree); frequency agrees with the sheet for mIPSC only |
| fig5b-sepsc-frequency | blocked, methods_underspecified | 5 pA: VEN 2.06 < PC 9.36 Hz, p = 0.026; RMS: VEN 32.1 > PC 23.0 Hz, p = 0.091 |
| fig5b-sipsc-frequency | blocked, methods_underspecified | 5 pA: VEN 19.9 > PC 15.4 Hz, p = 0.34 to 0.40; RMS: VEN higher, p = 0.0006 to 0.072; mostly noise |
| fig5b-mepsc-frequency | not reproduced | table labels, 5 pA: VEN 0.92 > PC 0.69 Hz, p = 0.61; RMS VEN higher, p = 0.15 |
| fig5b-mipsc-frequency | blocked, methods_underspecified | 5 pA: VEN 1.23 < PC 2.35 Hz, p = 0.0074; RMS: p = 0.24 |
| fig5c-sepsc-amplitude | reproduced | 8.20 against 8.51 pA, p = 0.24 |
| fig5c-sipsc-amplitude | blocked, methods_underspecified | 11.8 against 14.4 pA, p = 0.094 |
| fig5c-mepsc-amplitude | reproduced | 8.55 against 7.81 pA, p = 0.61 |
| fig5c-mipsc-amplitude | reproduced | 9.57 against 10.20 pA, p = 0.46 |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's. The paper detected events in miniAnalysis and measured them
in Clampfit, with settings it does not give beyond a 5 pA threshold; here a detector of the same scheme
with standard parameters is used (helper docstring). The detector check shows where it can be trusted: in
files with noise below 2 pA (all mEPSC, 34 of 36 mIPSC, 10 of 18 sEPSC files) it recovers 91 to 94% of
injected events and finds few opposite-polarity peaks; in the sEPSC files with noise of 2 to 4 pA
the opposite-polarity peaks are about half as frequent as the events (0.76 to 1.00 in the three
sEPSC files above 4 pA), and in 13 of the 15 sIPSC files
0.87 to 1.54 times as frequent (about half in the other two), so a large part of the 5 pA frequencies there
is noise, and they are several times the code's sheets. For this reason the two sIPSC claims are
blocked (methods_underspecified).
The threshold at the recording's RMS noise, reading (b) of the claims
file, finds as many opposite-polarity peaks as events in every experiment; a question to a reader asks
whether the claims file misreads the Methods there. The analysed stretch is the whole deposited recording
(the code's mIPSC sheet used 30 s, shown as a variant). Cell types come from the code's table, not from the
dataset. The paper's own sheet values, tested as the figure tests them, give VEN lower in all four
frequencies (p = 0.0068 to 0.028) with the sheet's labels; with the dataset's labels the mEPSC order
reverses.

What was found about the dataset.
- The NWB files carry no cell type; the type of each file is only in the paper's code (metadata_PSC.csv).
- That table names one sIPSC file twice (as a PC and as a VEN) and does not name `fwphyw`; and for every
  mEPSC cell its label is the opposite of the paper's analysis sheet. The dandiset holds 15 mEPSC files
  where the paper reports 16 cells.
- The spontaneous IPSC files are described as held at +20 mV and their holding currents are positive
  (107 to 979 pA), where the Methods state IPSCs were held at -60 mV.
- 22 of 36 mIPSC files end in runs of exact zeros (6,795 to 30,167 samples); 44 of 84 files are shorter
  than the 5 min the Methods give as the minimum (sEPSC 14 of 18, sIPSC 8 of 15, mIPSC 22 of 36 after the
  padding is cut).
- The noise of 11 of 15 sIPSC files and 2 sEPSC files exceeds the 5 pA the Methods give as the largest
  RMS noise; no sIPSC file has noise below 2 pA.
- All files have rate 1000 Hz; the decay times of the average events (sEPSC median 7 ms, sIPSC 14 ms,
  mEPSC 9 ms, mIPSC 16 ms) are consistent with it. There are no trial tables or stimuli, so no other clock
  check applies.
- No two files are identical, and no pair correlates above 0.15 after the drift is removed.
- `cell_id` is not unique within a session (three pairs), so it does not identify a cell.